# 🧪 HƯỚNG DẪN SỬ DỤNG 4 DATASET & QUY TRÌNH CHIA TRAIN / VAL / TEST DÙNG CHUNG CHO CẢ NHÓM (3 THÀNH VIÊN)
* **Ngày cập nhật:** 2026-10-21
* **Mô hình Backbone:** ViT-B/16 Frozen (áp dụng đồng nhất cho cả hướng Gradient-based và hướng Nghiệm đóng Analytic Ridge)
* **Mục tiêu:** Chuẩn hóa 100% cách truy cập dữ liệu và quy trình đánh giá thực nghiệm giữa 3 thành viên trong nhóm. Thành viên làm dataset nào chỉ cần **copy nguyên Cell tương ứng bên dưới** vào notebook của mình là chạy được ngay, không cần tạo file CSV trung gian hay tự viết lại hàm chia dữ liệu.

---

## 📌 1. ĐƯỜNG DẪN 4 DATASET TRÊN KAGGLE (ĐÃ CHIA SẴN TRAIN / TEST)
Các bộ dữ liệu trên Kaggle đều đã được chia sẵn thành 2 phần `train` và `test` chuẩn theo bài báo gốc (APT / RanPAC / DualPrompt):
1. **CIFAR-100** (100 lớp — 10 Tasks $\times$ 10 lớp | Train: `50,000` ảnh, Test: `10,000` ảnh):
   - Đường dẫn Kaggle: `[/kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python](https://www.kaggle.com/datasets/thohintrnhngc/cifar-100-python)`
2. **CUB-200-2011** (200 lớp — 10 Tasks $\times$ 20 lớp | Train: `5,994` ảnh, Test: `5,794` ảnh):
   - Đường dẫn Kaggle: `[/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder](https://www.kaggle.com/datasets/cyizhuo/cub-200-2011-by-classes-folder)`
3. **Stanford Cars-196** (196 lớp — 10 Tasks: Task 1 gồm 16 lớp + 9 Tasks $\times$ 20 lớp | Train: `8,144` ảnh, Test: `8,041` ảnh):
   - Đường dẫn Kaggle: `[/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder](https://www.kaggle.com/datasets/jutrera/stanford-car-dataset-by-classes-folder)`
4. **ImageNet-R** (200 lớp — 10 Tasks $\times$ 20 lớp | Train: `24,000` ảnh, Test: `6,000` ảnh):
   - Đường dẫn Kaggle: `[/kaggle/input/datasets/thohintrnhngc/imagenet-r](https://www.kaggle.com/datasets/thohintrnhngc/imagenet-r)`

---

## 🚨 2. QUY TRÌNH THỰC NGHIỆM 2 GIAI ĐOẠN BẮT BUỘC (CHỐNG RÒ RỈ TẬP TEST)

> [!IMPORTANT]
> **Nguyên tắc tối thượng:** Trong suốt quá trình cải tiến (improve), thử nghiệm kiến trúc (ablation), chọn mô hình và tinh chỉnh siêu tham số, **TUYỆT ĐỐI KHÔNG ĐƯỢC SỬ DỤNG TẬP TEST**.

### 🔹 Giai đoạn 1: Khi Improve & Chọn Model (Đánh giá trên tập `Val`)
* Tập `train` gốc được chia theo tỉ lệ **80% Train / 20% Val** theo **3 seed `0, 1, 2`**.
* Vì một số dataset có số lượng ảnh mỗi lớp nhỏ (như CUB chỉ ~30 ảnh/lớp, Cars ~42 ảnh/lớp), để kết quả đáng tin cậy nhất, **mỗi seed $s \in \{0, 1, 2\}$ sẽ đồng thời thực hiện 2 việc (Phương án A)**:
  1. **Đảo thứ tự các class vào 10 Tasks** theo `seed = s` (`np.random.RandomState(s).permutation(num_classes)`).
  2. **Đổi tập ảnh Val không trùng lặp (Stratified Disjoint 5-Fold, `base_seed = 1993`):**
     - Mỗi lớp trong tập `train` gốc được chia đều thành 5 phần (Fold 0, 1, 2, 3, 4 — mỗi phần đúng 20% ảnh của lớp đó).
     - `seed = 0`: Lấy **Fold 0** làm Val (20%), 4 fold còn lại (1, 2, 3, 4) làm Train (80%).
     - `seed = 1`: Lấy **Fold 1** làm Val (20%), 4 fold còn lại (0, 2, 3, 4) làm Train (80%).
     - `seed = 2`: Lấy **Fold 2** làm Val (20%), 4 fold còn lại (0, 1, 3, 4) làm Train (80%).
     - $\Rightarrow$ **Các tập Val của 3 seed hoàn toàn không trùng ảnh nào (Overlap = 0%)**, giúp qua 3 seed mô hình được kiểm chứng trên **60% toàn bộ dữ liệu Train** mà vẫn chỉ cần chạy đúng 3 seed!
* **Cách dùng trong code:** Gọi `get_task_subsets(..., phase="val")` để lấy `(train_80_subset, val_20_seen_subset)` cho từng Task.

### 🔹 Giai đoạn 2: Khi Đã Chốt Model Cuối Cùng (Train lại trên `100% Train gốc` & Đánh giá trên `Test`)
* Sau khi đã tìm được mô hình và bộ siêu tham số tốt nhất từ Giai đoạn 1 (dựa trên điểm Val trung bình 3 seed):
* **Cố định hoàn toàn** mô hình và siêu tham số đó (không chỉnh sửa nữa).
* **Train lại trên toàn bộ 100% tập `train` gốc (gộp cả 80% Train + 20% Val)** và **đánh giá trên tập `test` chính thức** theo **3 seed `0, 1, 2`** (ứng với 3 thứ tự task của seed `0, 1, 2`) để lấy số liệu cuối cùng đưa vào báo cáo / bài báo.
* **Cách dùng trong code:** Chỉ cần đổi đúng 1 tham số thành `get_task_subsets(..., phase="test")` để lấy `(full_train_100_subset, official_test_seen_subset)`.

---

**Cấu trúc Notebook này:** Dưới đây là **4 Cell hướng dẫn cụ thể và độc lập cho từng dataset** (kiểm tra thống kê ảnh, phân bố lớp, và chạy thử chia `Train 80% / Val 20%` theo 3 seed `0, 1, 2` trong quá trình Improve).


In [1]:
# ==============================================================================
# CELL ĐỘC LẬP 1: CIFAR-100 (Copy nguyên cell này vào notebook của bạn để dùng ngay)
# Đường dẫn chuẩn: /kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python
# ==============================================================================
import os
import pickle
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, Subset

# 1. Tự động nhận diện đường dẫn CIFAR-100 trên Kaggle
CIFAR_CANDIDATES = [
    "/kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python",
]
CIFAR_ROOT = next((p for p in CIFAR_CANDIDATES if os.path.isdir(p) and os.path.isfile(os.path.join(p, "train"))), CIFAR_CANDIDATES[0])

class CIFAR100PickleDataset(Dataset):
    """Đọc trực tiếp file pickle train/test/meta có sẵn trên Kaggle (không cần tải lại)."""
    def __init__(self, root, train=True, transform=None):
        self.root = root
        self.transform = transform
        split_file = "train" if train else "test"
        with open(os.path.join(root, split_file), "rb") as f:
            entry = pickle.load(f, encoding="latin1")
        with open(os.path.join(root, "meta"), "rb") as f:
            meta = pickle.load(f, encoding="latin1")
        self.data = entry["data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)  # (N, 32, 32, 3) uint8
        self.targets = [int(y) for y in entry["fine_labels"]]
        self.classes = list(meta["fine_label_names"])

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img = Image.fromarray(self.data[idx])
        if self.transform is not None:
            img = self.transform(img)
        return img, self.targets[idx]

# 2. Hàm chuẩn Phương án A: vừa đảo thứ tự Task theo seed, vừa lấy Fold[seed] làm Val (0% overlap)
def setup_cifar100_seed(train_dataset, seed, num_classes=100, task_sizes=None, n_splits=5, base_seed=1993):
    assert 0 <= seed < n_splits, f"seed phải nằm trong [0, {n_splits-1}]"
    if task_sizes is None:
        task_sizes = [10] * 10  # 10 tasks x 10 classes = 100 classes

    # (a) Đảo thứ tự class vào 10 Tasks theo seed
    rng_task = np.random.RandomState(seed)
    class_order = rng_task.permutation(num_classes).tolist()
    task_classes, cur = [], 0
    for sz in task_sizes:
        task_classes.append(class_order[cur : cur + sz])
        cur += sz

    # (b) Chia Stratified Disjoint 5-Fold bên trong từng class -> lấy Fold[seed] làm Val (20%)
    targets = np.asarray(train_dataset.targets)
    tr_idx, va_idx = [], []
    rng_split = np.random.RandomState(base_seed)
    for c in range(num_classes):
        c_indices = np.where(targets == c)[0]
        perm = rng_split.permutation(c_indices)
        folds = np.array_split(perm, n_splits)
        va_idx.extend(folds[seed])
        for k in range(n_splits):
            if k != seed:
                tr_idx.extend(folds[k])

    return task_classes, np.array(tr_idx), np.array(va_idx)

def get_task_subsets(train_dataset, test_dataset, task_classes, tr_idx, va_idx, task_id, phase="val"):
    """
    Lấy Subset tại Task `task_id` (0..9):
      - phase="val"  (Giai đoạn 1 - Improve/Chọn model): học trên 80% Train, đánh giá trên 20% Val (KHÔNG đụng Test).
      - phase="test" (Giai đoạn 2 - Chốt model cuối)   : học trên 100% Train gốc (cả Val + Train), đánh giá trên Test.
    """
    cur_cls = set(task_classes[task_id])
    seen_cls = set(sum(task_classes[: task_id + 1], []))
    tr_targets = np.asarray(train_dataset.targets)

    if phase == "val":
        t_tr_idx = [i for i in tr_idx if tr_targets[i] in cur_cls]
        t_ev_idx = [i for i in va_idx if tr_targets[i] in seen_cls]
        return Subset(train_dataset, t_tr_idx), Subset(train_dataset, t_ev_idx)
    elif phase == "test":
        te_targets = np.asarray(test_dataset.targets)
        t_tr_idx = [i for i in range(len(tr_targets)) if tr_targets[i] in cur_cls]
        t_ev_idx = [i for i in range(len(te_targets)) if te_targets[i] in seen_cls]
        return Subset(train_dataset, t_tr_idx), Subset(test_dataset, t_ev_idx)
    else:
        raise ValueError("phase phải là 'val' hoặc 'test'")

# 3. Chạy kiểm tra dữ liệu & In thống kê chia Train/Val theo 3 seed [0, 1, 2]
cifar_train = CIFAR100PickleDataset(CIFAR_ROOT, train=True)
cifar_test  = CIFAR100PickleDataset(CIFAR_ROOT, train=False)

tr_counts = pd.Series(cifar_train.targets).value_counts().sort_index()
te_counts = pd.Series(cifar_test.targets).value_counts().sort_index()

print("=" * 100)
print(f"✅ [1. CIFAR-100] Đọc từ: {CIFAR_ROOT}")
print(f"   Số lớp: {len(cifar_train.classes)} | Train gốc (100%): {len(cifar_train)} (Paper: 50000) | Test (Đóng băng): {len(cifar_test)} (Paper: 10000)")
print(f"   Phân bố ảnh/lớp (Train gốc): Min={tr_counts.min()}, Max={tr_counts.max()}, Mean={tr_counts.mean():.2f}, Std={tr_counts.std():.2f}")
print(f"   Phân bố ảnh/lớp (Test)     : Min={te_counts.min()}, Max={te_counts.max()}, Mean={te_counts.mean():.2f}, Std={te_counts.std():.2f}")
print("-" * 100)

val_sets, seed_rows = {}, []
for s in [0, 1, 2]:
    t_cls, tr_i, va_i = setup_cifar100_seed(cifar_train, seed=s)
    val_sets[s] = set(va_i.tolist())
    va_per_cls = pd.Series(np.asarray(cifar_train.targets)[va_i]).value_counts()
    seed_rows.append({
        "Seed": s, "Train 80% (Improve)": len(tr_i), "Val 20% (Improve)": len(va_i),
        "Min Val/lớp": int(va_per_cls.min()), "Max Val/lớp": int(va_per_cls.max()),
        "Train 100% (Chốt)": len(cifar_train), "Test (Chốt)": len(cifar_test),
        "Task 1 (5 lớp đầu)": t_cls[0][:5]
    })

print(pd.DataFrame(seed_rows).to_string(index=False))
ov01, ov12, ov02 = len(val_sets[0] & val_sets[1]), len(val_sets[1] & val_sets[2]), len(val_sets[0] & val_sets[2])
unq_val = len(val_sets[0] | val_sets[1] | val_sets[2])
print(f"👉 Kiểm chứng Val 3 seeds không trùng: Overlap(0,1)={ov01} | Overlap(1,2)={ov12} | Overlap(0,2)={ov02}")
print(f"👉 Tổng ảnh Val DUY NHẤT qua 3 seed : {unq_val}/{len(cifar_train)} ({100*unq_val/len(cifar_train):.1f}% tập Train gốc)")
print("=" * 100)


✅ [1. CIFAR-100] Đọc từ: /kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python
   Số lớp: 100 | Train gốc (100%): 50000 (Paper: 50000) | Test (Đóng băng): 10000 (Paper: 10000)
   Phân bố ảnh/lớp (Train gốc): Min=500, Max=500, Mean=500.00, Std=0.00
   Phân bố ảnh/lớp (Test)     : Min=100, Max=100, Mean=100.00, Std=0.00
----------------------------------------------------------------------------------------------------
 Seed  Train 80% (Improve)  Val 20% (Improve)  Min Val/lớp  Max Val/lớp  Train 100% (Chốt)  Test (Chốt)   Task 1 (5 lớp đầu)
    0                40000              10000          100          100              50000        10000  [26, 86, 2, 55, 75]
    1                40000              10000          100          100              50000        10000 [80, 84, 33, 81, 93]
    2                40000              10000          100          100              50000        10000 [83, 30, 56, 24, 16]
👉 Kiểm chứng Val 3 seeds không trùng: Overlap(0,1)=0 | Overlap

In [2]:
# ==============================================================================
# CELL ĐỘC LẬP 2: CUB-200-2011 (Copy nguyên cell này vào notebook của bạn để dùng ngay)
# Đường dẫn chuẩn: /kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder
# ==============================================================================
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Subset
from torchvision.datasets import ImageFolder

# 1. Tự động nhận diện đường dẫn CUB-200-2011 trên Kaggle
CUB_CANDIDATES = [
    "/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder",
]
CUB_ROOT = next((p for p in CUB_CANDIDATES if os.path.isdir(os.path.join(p, "train")) and os.path.isdir(os.path.join(p, "test"))), CUB_CANDIDATES[0])

# 2. Hàm chuẩn Phương án A: vừa đảo thứ tự Task theo seed, vừa lấy Fold[seed] làm Val (0% overlap)
def setup_cub200_seed(train_dataset, seed, num_classes=200, task_sizes=None, n_splits=5, base_seed=1993):
    assert 0 <= seed < n_splits, f"seed phải nằm trong [0, {n_splits-1}]"
    if task_sizes is None:
        task_sizes = [20] * 10  # 10 tasks x 20 classes = 200 classes

    # (a) Đảo thứ tự class vào 10 Tasks theo seed
    rng_task = np.random.RandomState(seed)
    class_order = rng_task.permutation(num_classes).tolist()
    task_classes, cur = [], 0
    for sz in task_sizes:
        task_classes.append(class_order[cur : cur + sz])
        cur += sz

    # (b) Chia Stratified Disjoint 5-Fold bên trong từng class -> lấy Fold[seed] làm Val (20%)
    targets = np.asarray(train_dataset.targets)
    tr_idx, va_idx = [], []
    rng_split = np.random.RandomState(base_seed)
    for c in range(num_classes):
        c_indices = np.where(targets == c)[0]
        perm = rng_split.permutation(c_indices)
        folds = np.array_split(perm, n_splits)
        va_idx.extend(folds[seed])
        for k in range(n_splits):
            if k != seed:
                tr_idx.extend(folds[k])

    return task_classes, np.array(tr_idx), np.array(va_idx)

def get_task_subsets(train_dataset, test_dataset, task_classes, tr_idx, va_idx, task_id, phase="val"):
    """
    Lấy Subset tại Task `task_id` (0..9):
      - phase="val"  (Giai đoạn 1 - Improve/Chọn model): học trên 80% Train, đánh giá trên 20% Val (KHÔNG đụng Test).
      - phase="test" (Giai đoạn 2 - Chốt model cuối)   : học trên 100% Train gốc (cả Val + Train), đánh giá trên Test.
    """
    cur_cls = set(task_classes[task_id])
    seen_cls = set(sum(task_classes[: task_id + 1], []))
    tr_targets = np.asarray(train_dataset.targets)

    if phase == "val":
        t_tr_idx = [i for i in tr_idx if tr_targets[i] in cur_cls]
        t_ev_idx = [i for i in va_idx if tr_targets[i] in seen_cls]
        return Subset(train_dataset, t_tr_idx), Subset(train_dataset, t_ev_idx)
    elif phase == "test":
        te_targets = np.asarray(test_dataset.targets)
        t_tr_idx = [i for i in range(len(tr_targets)) if tr_targets[i] in cur_cls]
        t_ev_idx = [i for i in range(len(te_targets)) if te_targets[i] in seen_cls]
        return Subset(train_dataset, t_tr_idx), Subset(test_dataset, t_ev_idx)
    else:
        raise ValueError("phase phải là 'val' hoặc 'test'")

# 3. Chạy kiểm tra dữ liệu & In thống kê chia Train/Val theo 3 seed [0, 1, 2]
cub_train = ImageFolder(os.path.join(CUB_ROOT, "train"))
cub_test  = ImageFolder(os.path.join(CUB_ROOT, "test"))

tr_counts = pd.Series(cub_train.targets).value_counts().sort_index()
te_counts = pd.Series(cub_test.targets).value_counts().sort_index()

print("=" * 100)
print(f"✅ [2. CUB-200-2011] Đọc từ: {CUB_ROOT}")
print(f"   Số lớp: {len(cub_train.classes)} | Train gốc (100%): {len(cub_train)} (Paper: 5994) | Test (Đóng băng): {len(cub_test)} (Paper: 5794)")
print(f"   Phân bố ảnh/lớp (Train gốc): Min={tr_counts.min()}, Max={tr_counts.max()}, Mean={tr_counts.mean():.2f}, Std={tr_counts.std():.2f}")
print(f"   Phân bố ảnh/lớp (Test)     : Min={te_counts.min()}, Max={te_counts.max()}, Mean={te_counts.mean():.2f}, Std={te_counts.std():.2f}")
print("-" * 100)

val_sets, seed_rows = {}, []
for s in [0, 1, 2]:
    t_cls, tr_i, va_i = setup_cub200_seed(cub_train, seed=s)
    val_sets[s] = set(va_i.tolist())
    va_per_cls = pd.Series(np.asarray(cub_train.targets)[va_i]).value_counts()
    seed_rows.append({
        "Seed": s, "Train 80% (Improve)": len(tr_i), "Val 20% (Improve)": len(va_i),
        "Min Val/lớp": int(va_per_cls.min()), "Max Val/lớp": int(va_per_cls.max()),
        "Train 100% (Chốt)": len(cub_train), "Test (Chốt)": len(cub_test),
        "Task 1 (5 lớp đầu)": t_cls[0][:5]
    })

print(pd.DataFrame(seed_rows).to_string(index=False))
ov01, ov12, ov02 = len(val_sets[0] & val_sets[1]), len(val_sets[1] & val_sets[2]), len(val_sets[0] & val_sets[2])
unq_val = len(val_sets[0] | val_sets[1] | val_sets[2])
print(f"👉 Kiểm chứng Val 3 seeds không trùng: Overlap(0,1)={ov01} | Overlap(1,2)={ov12} | Overlap(0,2)={ov02}")
print(f"👉 Tổng ảnh Val DUY NHẤT qua 3 seed : {unq_val}/{len(cub_train)} ({100*unq_val/len(cub_train):.1f}% tập Train gốc)")
print("=" * 100)


✅ [2. CUB-200-2011] Đọc từ: /kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder
   Số lớp: 200 | Train gốc (100%): 5994 (Paper: 5994) | Test (Đóng băng): 5794 (Paper: 5794)
   Phân bố ảnh/lớp (Train gốc): Min=29, Max=30, Mean=29.97, Std=0.17
   Phân bố ảnh/lớp (Test)     : Min=11, Max=30, Mean=28.97, Std=2.92
----------------------------------------------------------------------------------------------------
 Seed  Train 80% (Improve)  Val 20% (Improve)  Min Val/lớp  Max Val/lớp  Train 100% (Chốt)  Test (Chốt)       Task 1 (5 lớp đầu)
    0                 4794               1200            6            6               5994         5794  [18, 170, 107, 98, 177]
    1                 4794               1200            6            6               5994         5794   [58, 40, 34, 102, 184]
    2                 4794               1200            6            6               5994         5794 [112, 29, 182, 199, 193]
👉 Kiểm chứng Val 3 seeds không trùng: Overlap(0,1)=0 | Overlap

In [3]:
# ==============================================================================
# CELL ĐỘC LẬP 3: STANFORD CARS-196 (Copy nguyên cell này vào notebook của bạn để dùng ngay)
# Đường dẫn chuẩn: /kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder
# ==============================================================================
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Subset
from torchvision.datasets import ImageFolder

# 1. Tự động nhận diện đường dẫn Stanford Cars-196 trên Kaggle
CARS_CANDIDATES = [
    "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data/car_data",
]
CARS_ROOT = next((p for p in CARS_CANDIDATES if os.path.isdir(os.path.join(p, "train")) and os.path.isdir(os.path.join(p, "test"))), CARS_CANDIDATES[0])

# 2. Hàm chuẩn Phương án A: vừa đảo thứ tự Task theo seed, vừa lấy Fold[seed] làm Val (0% overlap)
def setup_cars196_seed(train_dataset, seed, num_classes=196, task_sizes=None, n_splits=5, base_seed=1993):
    assert 0 <= seed < n_splits, f"seed phải nằm trong [0, {n_splits-1}]"
    if task_sizes is None:
        task_sizes = [16] + [20] * 9  # Chuẩn APT/RanPAC cho Cars-196: Task 1 (16 lớp) + 9 Tasks (20 lớp) = 196 lớp

    # (a) Đảo thứ tự class vào 10 Tasks theo seed
    rng_task = np.random.RandomState(seed)
    class_order = rng_task.permutation(num_classes).tolist()
    task_classes, cur = [], 0
    for sz in task_sizes:
        task_classes.append(class_order[cur : cur + sz])
        cur += sz

    # (b) Chia Stratified Disjoint 5-Fold bên trong từng class -> lấy Fold[seed] làm Val (20%)
    targets = np.asarray(train_dataset.targets)
    tr_idx, va_idx = [], []
    rng_split = np.random.RandomState(base_seed)
    for c in range(num_classes):
        c_indices = np.where(targets == c)[0]
        perm = rng_split.permutation(c_indices)
        folds = np.array_split(perm, n_splits)
        va_idx.extend(folds[seed])
        for k in range(n_splits):
            if k != seed:
                tr_idx.extend(folds[k])

    return task_classes, np.array(tr_idx), np.array(va_idx)

def get_task_subsets(train_dataset, test_dataset, task_classes, tr_idx, va_idx, task_id, phase="val"):
    """
    Lấy Subset tại Task `task_id` (0..9):
      - phase="val"  (Giai đoạn 1 - Improve/Chọn model): học trên 80% Train, đánh giá trên 20% Val (KHÔNG đụng Test).
      - phase="test" (Giai đoạn 2 - Chốt model cuối)   : học trên 100% Train gốc (cả Val + Train), đánh giá trên Test.
    """
    cur_cls = set(task_classes[task_id])
    seen_cls = set(sum(task_classes[: task_id + 1], []))
    tr_targets = np.asarray(train_dataset.targets)

    if phase == "val":
        t_tr_idx = [i for i in tr_idx if tr_targets[i] in cur_cls]
        t_ev_idx = [i for i in va_idx if tr_targets[i] in seen_cls]
        return Subset(train_dataset, t_tr_idx), Subset(train_dataset, t_ev_idx)
    elif phase == "test":
        te_targets = np.asarray(test_dataset.targets)
        t_tr_idx = [i for i in range(len(tr_targets)) if tr_targets[i] in cur_cls]
        t_ev_idx = [i for i in range(len(te_targets)) if te_targets[i] in seen_cls]
        return Subset(train_dataset, t_tr_idx), Subset(test_dataset, t_ev_idx)
    else:
        raise ValueError("phase phải là 'val' hoặc 'test'")

# 3. Chạy kiểm tra dữ liệu & In thống kê chia Train/Val theo 3 seed [0, 1, 2]
cars_train = ImageFolder(os.path.join(CARS_ROOT, "train"))
cars_test  = ImageFolder(os.path.join(CARS_ROOT, "test"))

tr_counts = pd.Series(cars_train.targets).value_counts().sort_index()
te_counts = pd.Series(cars_test.targets).value_counts().sort_index()

print("=" * 100)
print(f"✅ [3. STANFORD CARS-196] Đọc từ: {CARS_ROOT}")
print(f"   Số lớp: {len(cars_train.classes)} | Train gốc (100%): {len(cars_train)} (Paper: 8144) | Test (Đóng băng): {len(cars_test)} (Paper: 8041)")
print(f"   Phân bố ảnh/lớp (Train gốc): Min={tr_counts.min()}, Max={tr_counts.max()}, Mean={tr_counts.mean():.2f}, Std={tr_counts.std():.2f}")
print(f"   Phân bố ảnh/lớp (Test)     : Min={te_counts.min()}, Max={te_counts.max()}, Mean={te_counts.mean():.2f}, Std={te_counts.std():.2f}")
print("-" * 100)

val_sets, seed_rows = {}, []
for s in [0, 1, 2]:
    t_cls, tr_i, va_i = setup_cars196_seed(cars_train, seed=s)
    val_sets[s] = set(va_i.tolist())
    va_per_cls = pd.Series(np.asarray(cars_train.targets)[va_i]).value_counts()
    seed_rows.append({
        "Seed": s, "Train 80% (Improve)": len(tr_i), "Val 20% (Improve)": len(va_i),
        "Min Val/lớp": int(va_per_cls.min()), "Max Val/lớp": int(va_per_cls.max()),
        "Train 100% (Chốt)": len(cars_train), "Test (Chốt)": len(cars_test),
        "Task 1 (5 lớp đầu)": t_cls[0][:5]
    })

print(pd.DataFrame(seed_rows).to_string(index=False))
ov01, ov12, ov02 = len(val_sets[0] & val_sets[1]), len(val_sets[1] & val_sets[2]), len(val_sets[0] & val_sets[2])
unq_val = len(val_sets[0] | val_sets[1] | val_sets[2])
print(f"👉 Kiểm chứng Val 3 seeds không trùng: Overlap(0,1)={ov01} | Overlap(1,2)={ov12} | Overlap(0,2)={ov02}")
print(f"👉 Tổng ảnh Val DUY NHẤT qua 3 seed : {unq_val}/{len(cars_train)} ({100*unq_val/len(cars_train):.1f}% tập Train gốc)")
print("=" * 100)


✅ [3. STANFORD CARS-196] Đọc từ: /kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data/car_data
   Số lớp: 196 | Train gốc (100%): 8144 (Paper: 8144) | Test (Đóng băng): 8041 (Paper: 8041)
   Phân bố ảnh/lớp (Train gốc): Min=24, Max=68, Mean=41.55, Std=4.33
   Phân bố ảnh/lớp (Test)     : Min=24, Max=68, Mean=41.03, Std=4.37
----------------------------------------------------------------------------------------------------
 Seed  Train 80% (Improve)  Val 20% (Improve)  Min Val/lớp  Max Val/lớp  Train 100% (Chốt)  Test (Chốt)      Task 1 (5 lớp đầu)
    0                 6442               1702            5           14               8144         8041  [83, 12, 33, 113, 171]
    1                 6478               1666            5           14               8144         8041 [127, 11, 110, 124, 18]
    2                 6509               1635            5           14               8144         8041 [10, 79, 165, 143, 163]
👉 Kiểm chứng Val 3 seeds không trùn

In [4]:
# ==============================================================================
# CELL ĐỘC LẬP 4: IMAGENET-R (Copy nguyên cell này vào notebook của bạn để dùng ngay)
# Đường dẫn chuẩn: /kaggle/input/datasets/thohintrnhngc/imagenet-r
# ==============================================================================
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Subset
from torchvision.datasets import ImageFolder

# 1. Tự động nhận diện đường dẫn ImageNet-R trên Kaggle
IMNET_R_CANDIDATES = [
    "/kaggle/input/datasets/thohintrnhngc/imagenet-r",
]
IMNET_R_ROOT = next((p for p in IMNET_R_CANDIDATES if os.path.isdir(os.path.join(p, "train")) and os.path.isdir(os.path.join(p, "test"))), IMNET_R_CANDIDATES[0])

# 2. Hàm chuẩn Phương án A: vừa đảo thứ tự Task theo seed, vừa lấy Fold[seed] làm Val (0% overlap)
def setup_imagenet_r_seed(train_dataset, seed, num_classes=200, task_sizes=None, n_splits=5, base_seed=1993):
    assert 0 <= seed < n_splits, f"seed phải nằm trong [0, {n_splits-1}]"
    if task_sizes is None:
        task_sizes = [20] * 10  # 10 tasks x 20 classes = 200 classes

    # (a) Đảo thứ tự class vào 10 Tasks theo seed
    rng_task = np.random.RandomState(seed)
    class_order = rng_task.permutation(num_classes).tolist()
    task_classes, cur = [], 0
    for sz in task_sizes:
        task_classes.append(class_order[cur : cur + sz])
        cur += sz

    # (b) Chia Stratified Disjoint 5-Fold bên trong từng class -> lấy Fold[seed] làm Val (20%)
    targets = np.asarray(train_dataset.targets)
    tr_idx, va_idx = [], []
    rng_split = np.random.RandomState(base_seed)
    for c in range(num_classes):
        c_indices = np.where(targets == c)[0]
        perm = rng_split.permutation(c_indices)
        folds = np.array_split(perm, n_splits)
        va_idx.extend(folds[seed])
        for k in range(n_splits):
            if k != seed:
                tr_idx.extend(folds[k])

    return task_classes, np.array(tr_idx), np.array(va_idx)

def get_task_subsets(train_dataset, test_dataset, task_classes, tr_idx, va_idx, task_id, phase="val"):
    """
    Lấy Subset tại Task `task_id` (0..9):
      - phase="val"  (Giai đoạn 1 - Improve/Chọn model): học trên 80% Train, đánh giá trên 20% Val (KHÔNG đụng Test).
      - phase="test" (Giai đoạn 2 - Chốt model cuối)   : học trên 100% Train gốc (cả Val + Train), đánh giá trên Test.
    """
    cur_cls = set(task_classes[task_id])
    seen_cls = set(sum(task_classes[: task_id + 1], []))
    tr_targets = np.asarray(train_dataset.targets)

    if phase == "val":
        t_tr_idx = [i for i in tr_idx if tr_targets[i] in cur_cls]
        t_ev_idx = [i for i in va_idx if tr_targets[i] in seen_cls]
        return Subset(train_dataset, t_tr_idx), Subset(train_dataset, t_ev_idx)
    elif phase == "test":
        te_targets = np.asarray(test_dataset.targets)
        t_tr_idx = [i for i in range(len(tr_targets)) if tr_targets[i] in cur_cls]
        t_ev_idx = [i for i in range(len(te_targets)) if te_targets[i] in seen_cls]
        return Subset(train_dataset, t_tr_idx), Subset(test_dataset, t_ev_idx)
    else:
        raise ValueError("phase phải là 'val' hoặc 'test'")

# 3. Chạy kiểm tra dữ liệu & In thống kê chia Train/Val theo 3 seed [0, 1, 2]
imnet_train = ImageFolder(os.path.join(IMNET_R_ROOT, "train"))
imnet_test  = ImageFolder(os.path.join(IMNET_R_ROOT, "test"))

tr_counts = pd.Series(imnet_train.targets).value_counts().sort_index()
te_counts = pd.Series(imnet_test.targets).value_counts().sort_index()

print("=" * 100)
print(f"✅ [4. IMAGENET-R] Đọc từ: {IMNET_R_ROOT}")
print(f"   Số lớp: {len(imnet_train.classes)} | Train gốc (100%): {len(imnet_train)} (Paper: 24000) | Test (Đóng băng): {len(imnet_test)} (Paper: 6000)")
print(f"   Phân bố ảnh/lớp (Train gốc): Min={tr_counts.min()}, Max={tr_counts.max()}, Mean={tr_counts.mean():.2f}, Std={tr_counts.std():.2f}")
print(f"   Phân bố ảnh/lớp (Test)     : Min={te_counts.min()}, Max={te_counts.max()}, Mean={te_counts.mean():.2f}, Std={te_counts.std():.2f}")
print("-" * 100)

val_sets, seed_rows = {}, []
for s in [0, 1, 2]:
    t_cls, tr_i, va_i = setup_imagenet_r_seed(imnet_train, seed=s)
    val_sets[s] = set(va_i.tolist())
    va_per_cls = pd.Series(np.asarray(imnet_train.targets)[va_i]).value_counts()
    seed_rows.append({
        "Seed": s, "Train 80% (Improve)": len(tr_i), "Val 20% (Improve)": len(va_i),
        "Min Val/lớp": int(va_per_cls.min()), "Max Val/lớp": int(va_per_cls.max()),
        "Train 100% (Chốt)": len(imnet_train), "Test (Chốt)": len(imnet_test),
        "Task 1 (5 lớp đầu)": t_cls[0][:5]
    })

print(pd.DataFrame(seed_rows).to_string(index=False))
ov01, ov12, ov02 = len(val_sets[0] & val_sets[1]), len(val_sets[1] & val_sets[2]), len(val_sets[0] & val_sets[2])
unq_val = len(val_sets[0] | val_sets[1] | val_sets[2])
print(f"👉 Kiểm chứng Val 3 seeds không trùng: Overlap(0,1)={ov01} | Overlap(1,2)={ov12} | Overlap(0,2)={ov02}")
print(f"👉 Tổng ảnh Val DUY NHẤT qua 3 seed : {unq_val}/{len(imnet_train)} ({100*unq_val/len(imnet_train):.1f}% tập Train gốc)")
print("=" * 100)


✅ [4. IMAGENET-R] Đọc từ: /kaggle/input/datasets/thohintrnhngc/imagenet-r
   Số lớp: 200 | Train gốc (100%): 24000 (Paper: 24000) | Test (Đóng băng): 6000 (Paper: 6000)
   Phân bố ảnh/lớp (Train gốc): Min=38, Max=349, Mean=120.00, Std=58.75
   Phân bố ảnh/lớp (Test)     : Min=7, Max=81, Mean=30.00, Std=14.79
----------------------------------------------------------------------------------------------------
 Seed  Train 80% (Improve)  Val 20% (Improve)  Min Val/lớp  Max Val/lớp  Train 100% (Chốt)  Test (Chốt)       Task 1 (5 lớp đầu)
    0                19124               4876            8           70              24000         6000  [18, 170, 107, 98, 177]
    1                19162               4838            8           70              24000         6000   [58, 40, 34, 102, 184]
    2                19202               4798            8           70              24000         6000 [112, 29, 182, 199, 193]
👉 Kiểm chứng Val 3 seeds không trùng: Overlap(0,1)=0 | Overlap(1,2)=0 | O